# Stage 1: Data Generation

Generates reasoning traces from GSM8K maths problems using LLaMA-3.1-8B via the Groq API.

Each problem is sent to the model which solves it step by step. The model's answer is compared to the correct answer to assign a label: 1 (correct) or 0 (wrong).

**Input:** GSM8K dataset (loaded from HuggingFace)  
**Output:** `data/gsm8k_with_traces.csv`

Before running: get a free Groq API key from https://console.groq.com and paste it in the cell below.

## Cell 1 — Install libraries
Run this first. The `-q` flag keeps output quiet.  
After this cell finishes → **Kernel → Restart** → then run all cells top to bottom.

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade groq httpx --user
print(" Done — now Kernel → Restart and run from Cell 2")

 Done — now Kernel → Restart and run from Cell 2


## Cell 2 — Imports and folder setup
Sets up all imports and creates the local `data/` folder where all files are saved.

In [ ]:
import os, re, time
import pandas as pd
import numpy as np
from groq import Groq
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# All output files saved here (created automatically)
OUTPUT_DIR      = os.path.join(os.getcwd(), 'data')
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'stage1_checkpoint_batch7.csv')
OUTPUT_FILE     = os.path.join(OUTPUT_DIR, 'gsm8k_with_traces_batch7.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f" Imports OK")
print(f" Saving to: {OUTPUT_DIR}")

## Cell 3 — Connect to Groq API
Paste your free API key from https://console.groq.com below.

In [ ]:
#  Paste your Groq API key here
GROQ_API_KEY = "YOUR_GROQ_API_KEY"  # replace this


 Groq connected — model: llama-3.1-8b-instant


## Cell 4 — Load GSM8K dataset
Downloads from HuggingFace — free, no login needed.

In [ ]:
print("Loading GSM8K...")
ds = load_dataset('gsm8k', 'main')
df_train = ds['train'].to_pandas()
df_test  = ds['test'].to_pandas()

print(f" Train: {len(df_train)} samples")
print(f" Test:  {len(df_test)} samples")
print()
print("Sample question:")
print(df_train['question'][0][:100], "...")

Loading GSM8K...
 Train: 7473 samples
 Test:  1319 samples

Sample question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m ...


## Cell 5 — Split data BEFORE running LLM
 **Critical:** Always split before the LLM runs. Splitting after risks data leakage.

| Group | Size | Purpose |
|---|---|---|
| Train | 700 (70%) | GNN learns from these |
| Validation | 150 (15%) | Monitor training |
| Test | 150 (15%) | Final evaluation only |

In [ ]:
TOTAL_SAMPLES = 1000
df_work = df_train.iloc[5000:6000].reset_index(drop=True)

df_tr, df_temp = train_test_split(df_work, test_size=0.30, random_state=42)
df_val, df_te  = train_test_split(df_temp, test_size=0.50, random_state=42)

df_tr['split']  = 'train'
df_val['split'] = 'val'
df_te['split']  = 'test'

df_all = pd.concat([df_tr, df_val, df_te]).reset_index(drop=True)

print(f"Train:      {len(df_tr)} samples")
print(f"Validation: {len(df_val)} samples")
print(f"Test:       {len(df_te)} samples")
print(f"Total:      {len(df_all)} samples")
print("\n Splits created")

Train:      700 samples
Validation: 150 samples
Test:       150 samples
Total:      1000 samples

 Splits created


## Cell 6 — Define prompt and helper functions

- **`ask_llm(question)`** — sends question to Llama-3.1-8b, gets step-by-step trace
- **`extract_number(text)`** — pulls final answer from `####` marker
- **`make_label(trace, gt)`** — returns `1` (correct) or `0` (wrong)

In [ ]:
SYSTEM_PROMPT = """You are a maths tutor.
Solve step by step. Start each step with 'Step N:'.
End with '#### answer' on the very last line. Just the number.
Example final line: #### 42"""


def ask_llm(question):
    """Send question to Llama-3.1-8B via Groq. Returns step-by-step trace."""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question}
        ],
        temperature=0.6,
        max_tokens=512
    )
    return response.choices[0].message.content


def extract_number(text):
    """Extract final answer number from #### marker."""
    if not text:
        return None
    m = re.search(r'####\s*([\d,\.]+)', text)
    if m:
        return m.group(1).replace(',', '').strip()
    nums = re.findall(r'\b\d+\.?\d*\b', text)
    return nums[-1] if nums else None


def make_label(trace, gt):
    """Compare LLM answer to ground truth. Returns 1 (correct) or 0 (wrong)."""
    pred = extract_number(trace)
    gt_n = extract_number(gt)
    if not pred or not gt_n:
        return None
    try:
        return 1 if float(pred) == float(gt_n) else 0
    except:
        return 0


# Quick test
print("Running quick test...")
test_trace = ask_llm("A shop sells 3 pens for $4. How much do 9 pens cost?")
print(test_trace)
print("Extracted:", extract_number(test_trace))
print("\n Functions working")

Running quick test...
Step 1: To find the cost of 9 pens, we need to determine the cost of one pen first. Since 3 pens cost $4, we can divide the total cost by the number of pens to find the cost of one pen.

Step 2: Divide $4 by 3 to find the cost of one pen. cost_per_pen = 4 / 3

Step 3: Now that we know the cost of one pen, we can multiply this by 9 to find the cost of 9 pens. total_cost = cost_per_pen * 9

Step 4: Calculate the cost of one pen: cost_per_pen = 1.33333333333

Step 5: Multiply the cost of one pen by 9 to get the total cost: total_cost = 1.33333333333 * 9

Step 6: Calculate the total cost: total_cost = 12

#### 12
Extracted: 12

 Functions working


## Cell 7 — Main generation loop (all 1,000 samples)

- `time.sleep(1)` between requests — avoids rate limit errors
- Checkpoint saved every 50 samples — safe to re-run if interrupted
- If kernel crashes, re-run this cell — it resumes automatically

**Expected time: ~17–20 minutes**  
If you get 429 errors on every row → your daily token limit is used up. Wait until midnight UTC and re-run.

In [ ]:
# Resume from checkpoint if one exists
if os.path.exists(CHECKPOINT_FILE):
    done_df  = pd.read_csv(CHECKPOINT_FILE)
    # Only count rows that actually have a trace (not failed rows)
    done_df  = done_df[done_df['trace'].notna()]
    done_idx = set(done_df['orig_idx'].tolist())
    results  = done_df.to_dict('records')
    print(f"  Resuming: {len(results)} valid rows already done")
else:
    done_idx = set()
    results  = []
    print(" Starting fresh run")

remaining = len(df_all) - len(done_idx)
print(f"   Samples remaining: {remaining}")
print(f"   Estimated time:    ~{remaining // 60 + 1} minutes")
print()

start_time = time.time()

for i, row in df_all.iterrows():
    if i in done_idx:
        continue

    try:
        time.sleep(1)  # do NOT remove — prevents 429 errors
        trace = ask_llm(row['question'])
        label = make_label(trace, row['answer'])

        results.append({
            'orig_idx':  i,
            'split':     row['split'],
            'question':  row['question'],
            'gt_answer': row['answer'],
            'trace':     trace,
            'pred':      extract_number(trace),
            'gt':        extract_number(row['answer']),
            'label':     label
        })

    except Exception as e:
        print(f"  Error at row {i}: {e}")
        # Do NOT save failed rows to checkpoint — skip them so they get retried
        continue

    # Save checkpoint every 50 samples
    if len(results) % 50 == 0:
        elapsed  = time.time() - start_time
        new_done = len(results) - (len(done_idx))
        per_item = elapsed / max(new_done, 1)
        eta      = (len(df_all) - len(results)) * per_item
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"[{len(results)}/{len(df_all)}] {elapsed/60:.1f}m elapsed  ETA: {eta/60:.1f}m — saved")

# Final save
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_FILE, index=False)

total_time = time.time() - start_time
print(f"\n DONE in {total_time/60:.1f} minutes")
print(f"   Saved to: {OUTPUT_FILE}")
print(f"   Total rows: {len(df_out)}")
print(f"   Label counts: {df_out['label'].value_counts().to_dict()}")

## Cell 8 — Balance the dataset

Llama-3.1-8b scores ~94% on GSM8K — too accurate, giving very few wrong traces.
We fix this by **downsampling the correct traces** to get a 75/25 balance.

This is standard ML practice — the traces are all real, we're just balancing classes.
The imbalance will also be handled in Stage 5 with class weights in the loss function.

**This cell reads the CSV, balances it, and saves it back. Run once after Cell 7 completes.**

In [ ]:
df = pd.read_csv(OUTPUT_FILE)

correct = df[df['label'] == 1]
wrong   = df[df['label'] == 0]

print(f"Before balancing: {len(correct)} correct ({len(correct)/len(df)*100:.1f}%), {len(wrong)} wrong ({len(wrong)/len(df)*100:.1f}%)")

# Keep all wrong traces + 3x that many correct traces = 75/25 split
n_correct_keep = min(len(wrong) * 3, len(correct))  # never try to sample more than available
correct_sampled = correct.sample(n=n_correct_keep, random_state=42)
df_balanced = pd.concat([correct_sampled, wrong]).sample(frac=1, random_state=42).reset_index(drop=True)

# Re-apply train/val/test split on balanced data
df_tr, df_temp = train_test_split(df_balanced, test_size=0.30, random_state=42)
df_val, df_te  = train_test_split(df_temp,     test_size=0.50, random_state=42)
df_tr['split']  = 'train'
df_val['split'] = 'val'
df_te['split']  = 'test'
df_final = pd.concat([df_tr, df_val, df_te]).reset_index(drop=True)

df_final.to_csv(OUTPUT_FILE, index=False)

n_c = len(df_final[df_final['label']==1])
n_w = len(df_final[df_final['label']==0])
print(f"After balancing:  {n_c} correct ({n_c/len(df_final)*100:.1f}%), {n_w} wrong ({n_w/len(df_final)*100:.1f}%)")
print(f"Total samples:    {len(df_final)}")
print(f"Train: {len(df_tr)}  Val: {len(df_val)}  Test: {len(df_te)}")
print("\n Balanced dataset saved — run Cell 9 quality checks")

Before balancing: 912 correct (91.5%), 85 wrong (8.5%)
After balancing:  255 correct (75.0%), 85 wrong (25.0%)
Total samples:    340
Train: 238  Val: 51  Test: 51

 Balanced dataset saved — run Cell 9 quality checks


## Cell 9 — Quality checks

All 6 checks must pass before moving to Stage 2.

| # | Check | Target |
|---|---|---|
| 1 | Total rows | > 0 |
| 2 | Missing traces | 0 |
| 3 | Label balance | 20–80% correct |
| 4 | Split sizes | 70/15/15 ratio |
| 5 | Step format | > 70% have 'Step N:' |
| 6 | Average steps | ≥ 2 per trace |

In [ ]:
df_out = pd.read_csv(OUTPUT_FILE)

print("=" * 55)
print("  STAGE 1 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: Row count
print(f"\n[1] Total rows: {len(df_out)}")
if len(df_out) > 0:
    print("     PASS")
else:
    print("     FAIL — no rows found")
    all_pass = False

# Check 2: Missing traces
missing = df_out['trace'].isna().sum()
print(f"\n[2] Missing traces: {missing}")
if missing == 0:
    print("     PASS")
else:
    print(f"      {missing} missing — re-run Cell 7 to fill them")
    all_pass = False

# Check 3: Label balance
counts        = df_out['label'].value_counts()
total_labeled = counts.sum()
pct_correct   = counts.get(1, 0) / total_labeled * 100
pct_wrong     = counts.get(0, 0) / total_labeled * 100
print(f"\n[3] Label balance:")
print(f"    Correct (1): {counts.get(1,0)} samples ({pct_correct:.1f}%)")
print(f"    Wrong   (0): {counts.get(0,0)} samples ({pct_wrong:.1f}%)")
if 20 <= pct_correct <= 80:
    print("     PASS — good balance")
else:
    print("      WARNING — run Cell 8 (balancing) first")
    all_pass = False

# Check 4: Split sizes
split_counts = df_out['split'].value_counts()
print(f"\n[4] Split sizes:")
for s in ['train', 'val', 'test']:
    c = split_counts.get(s, 0)
    print(f"    {s}: {c}")
print("     PASS")

# Check 5: Step format
has_steps = df_out['trace'].str.contains('Step', na=False).mean() * 100
print(f"\n[5] Traces with 'Step N:' format: {has_steps:.1f}%")
if has_steps >= 70:
    print("     PASS")
else:
    print("      WARNING — low step format")
    all_pass = False

# Check 6: Average steps
def count_steps(trace):
    if pd.isna(trace): return 0
    return len(re.findall(r'Step \d+:', trace))

df_out['num_steps'] = df_out['trace'].apply(count_steps)
avg_steps = df_out['num_steps'].mean()
print(f"\n[6] Average steps per trace: {avg_steps:.1f}")
print(f"    Min: {df_out['num_steps'].min()}  Max: {df_out['num_steps'].max()}")
if avg_steps >= 2:
    print("     PASS")
else:
    print("      WARNING — very few steps")
    all_pass = False

# Save with num_steps column added
df_out.to_csv(OUTPUT_FILE, index=False)

print()
print("=" * 55)
if all_pass:
    print("   ALL CHECKS PASSED — Stage 1 complete!")
    print("    Ready for Stage 2")
else:
    print("    SOME CHECKS FAILED — see above")
print("=" * 55)

  STAGE 1 QUALITY CHECKS

[1] Total rows: 340
     PASS

[2] Missing traces: 0
     PASS

[3] Label balance:
    Correct (1): 255 samples (75.0%)
    Wrong   (0): 85 samples (25.0%)
     PASS — good balance

[4] Split sizes:
    train: 238
    val: 51
    test: 51
     PASS

[5] Traces with 'Step N:' format: 100.0%
     PASS

[6] Average steps per trace: 4.9
    Min: 2  Max: 20
     PASS

   ALL CHECKS PASSED — Stage 1 complete!
    Ready for Stage 2


## Cell 10 — Manual inspection
Always read real examples before trusting numbers.

In [ ]:
df_out = pd.read_csv(OUTPUT_FILE)

def print_sample(row, label_text):
    print(f"Q: {row['question'][:90]}...")
    print(f"Trace:\n{row['trace'][:400]}")
    print(f"Predicted: {row['pred']}  |  GT: {row['gt']}  |  Label: {row['label']} ({label_text})")
    print("-" * 65)

print("3 CORRECT traces (label = 1)")
print("=" * 65)
for _, row in df_out[df_out['label'] == 1].head(3).iterrows():
    print_sample(row, "correct ")

print()
print("3 WRONG traces (label = 0)")
print("=" * 65)
for _, row in df_out[df_out['label'] == 0].head(3).iterrows():
    print_sample(row, "wrong ")

3 CORRECT traces (label = 1)
Q: For the family reunion, Peter is buying 16 pounds of bone-in chicken and half that amount ...
Trace:
Step 1: Calculate the amount of hamburgers Peter will buy, which is half the amount of chicken. 
16 pounds of chicken / 2 = 8 pounds of hamburgers.

Step 2: Calculate the amount of hot dogs Peter will buy, which is 2 pounds more than hamburgers.
8 pounds of hamburgers + 2 pounds = 10 pounds of hot dogs.

Step 3: Calculate the amount of sides Peter will buy, which is half the amount of hot dogs.
10
Predicted: 39.0  |  GT: 39  |  Label: 1 (correct )
-----------------------------------------------------------------
Q: Kylie and Kayla pick apples together and take home 340 apples total. If Kayla picked 10 mo...
Trace:
Step 1: Let's represent the number of apples Kylie picked as 'x'. 
Step 2: Since Kayla picked 10 more than 4 times the amount of apples Kylie picked, we can represent the number of apples Kayla picked as '4x + 10'.
Step 3: We know that the total

##  Stage 1 Complete!

**Output:** `data/gsm8k_with_traces.csv`

| Column | Contents |
|---|---|
| `orig_idx` | Original GSM8K row index |
| `split` | train / val / test |
| `question` | The maths problem |
| `gt_answer` | Ground truth answer from GSM8K |
| `trace` | Full step-by-step reasoning from LLM |
| `pred` | Number LLM gave as final answer |
| `gt` | Correct number from ground truth |
| `label` | 1 = correct, 0 = wrong |
| `num_steps` | Number of steps in trace |

**Next → Stage 2:** Parse each trace into individual steps → each step becomes a graph node.

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

DATA_DIR = 'data'

batches = [
    'gsm8k_with_traces_batch2.csv',
    'gsm8k_with_traces_batch3.csv',
    'gsm8k_with_traces_batch4.csv',
    'gsm8k_with_traces_batch5.csv',
    'gsm8k_with_traces_batch6.csv',
    'gsm8k_with_traces_batch7.csv',
]

dfs = []
for fname in batches:
    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    df = df[df['trace'].notna()].reset_index(drop=True)
    dfs.append(df)
    print(f'{fname}: {len(df)} rows')

combined = pd.concat(dfs, ignore_index=True)
combined = combined.drop_duplicates(subset=['question','trace']).reset_index(drop=True)
print(f'\nCombined: {len(combined)} rows')

correct = combined[combined['label']==1]
wrong   = combined[combined['label']==0]
n_keep  = min(len(wrong)*3, len(correct))
df_bal  = pd.concat([correct.sample(n=n_keep, random_state=42), wrong])
df_bal  = df_bal.sample(frac=1, random_state=42).reset_index(drop=True)

df_tr, df_temp = train_test_split(df_bal, test_size=0.30, random_state=42)
df_val, df_te  = train_test_split(df_temp, test_size=0.50, random_state=42)
df_tr['split']  = 'train'
df_val['split'] = 'val'
df_te['split']  = 'test'
df_final = pd.concat([df_tr, df_val, df_te]).reset_index(drop=True)

df_final.to_csv(os.path.join(DATA_DIR, 'gsm8k_with_traces.csv'), index=False)

print(f'\nFinal: {len(df_final)} traces')
print(f'Correct: {(df_final["label"]==1).sum()}  Wrong: {(df_final["label"]==0).sum()}')
print(f'Train: {len(df_tr)}  Val: {len(df_val)}  Test: {len(df_te)}')
print(f'Wrong in train: {(df_tr["label"]==0).sum()}')
print(' Saved to gsm8k_with_traces.csv')

gsm8k_with_traces_batch2.csv: 360 rows
gsm8k_with_traces_batch3.csv: 312 rows
gsm8k_with_traces_batch4.csv: 284 rows
gsm8k_with_traces_batch5.csv: 332 rows
gsm8k_with_traces_batch6.csv: 320 rows
gsm8k_with_traces_batch7.csv: 340 rows

Combined: 1948 rows

Final: 1948 traces
Correct: 1461  Wrong: 487
Train: 1363  Val: 292  Test: 293
Wrong in train: 343
 Saved to gsm8k_with_traces.csv
